# Tissue elevation heatmap -- investigation

Prototypes a per-pixel "elevation" map of tissue z-extent (a digital-
elevation-model-style heatmap, not the single scalar-per-FOV thickness
`after_imaging/08_measure_tissue_thickness.ipynb` already computes), plus a
z-sweep GIF of the downsampled, flat-field-corrected DAPI signal. This
notebook only investigates the approach on real data -- new reusable
functions stay local here; wiring anything into `MERci` proper (or into
`08_measure_tissue_thickness.ipynb`) is a follow-up session's task.

**Real test dataset**: `LT066_sample_01/merfish` (1138 real imaged FOVs,
101-z-step DAPI (405 nm) `cells` round, ST2 microscope) -- same dataset
`notebooks/tests/decrease_fov_number/01_*`/`02_*` already investigate, same
`SAMPLE_DIR`-explicit convention (too large to copy locally; read directly,
cache only computed/downsampled results).

**Algorithm** (see `prompt_history/` for the original request and this
notebook's first-pass review):
1. Identify **boundary FOVs** = the *exterior* FOVs of the imaged grid
   (`MERci.acquisition.positions.find_exterior_fovs` -- the same set
   `analysis/ffc.py`'s default `"exterior_grid"` strategy already selects
   for flat-field correction, since exterior FOVs are the ones assumed to
   be mostly tissue-free background).
2. Build the flat-field-correction (FFC) field -- **from every interior
   FOV's own full-z-stack min projection** (not boundary FOVs -- see the
   review note below), and estimate a background/foreground intensity
   threshold.
3. For a **small representative block of FOVs** (not the full 1138-FOV
   grid -- see the scope note below), and for every z-plane: FFC-correct,
   downsample (factor 8: 2304 -> 288 px), threshold, and record the
   elevation matrix `M` (the topmost z, in µm, at which each downsampled
   pixel is still foreground).
4. Crop each FOV's `M` to its non-overlap footprint and stitch the block's
   FOVs into one grid-indexed elevation heatmap.
5. Stitch the same block's downsampled DAPI images per z into a GIF.

**Scope note**: this real experiment's `cells` round is ~572 MB/FOV x 1138
FOVs (~650 GB); a full-grid run of step 3 is a cluster/SLURM-array job for
another session (`misc/measure_tissue_thickness_test.ipynb` sections
14/23/24 already have that SLURM-array pattern for similarly heavy per-FOV
z-stack loops). Steps 1-2 (boundary-FOV identification, FFC, threshold) DO
run over the real, full 1138-FOV grid below. Step 3 onward runs on one ~5x5
representative block only (chosen to straddle a real tissue-boundary/hole
edge, so it exercises both the FFC boundary-FOV logic and the
crop/stitch-across-neighbors logic).

**Review note (first pass -> this revision)**: two real issues found after
visually reviewing the first pass's real output:
1. **Every mosaic/heatmap below was built from raw, un-reoriented camera
   frames** -- this project has hit exactly this bug before
   (`notebooks/misc/correct_camera_rotation.ipynb`,
   `prompt_history/2026_07_31_1939_fix_camera_rotation_mosaic_never_
   oriented.md`): a raw frame's pixel grid does not match the real stage
   layout until `MERci.acquisition.merlin_config.apply_microscope_
   orientation` (transpose, then flip_horizontal, then flip_vertical) is
   applied -- otherwise a mosaic assembled from real stage positions
   looks scrambled/mirrored/rotated relative to the tissue's real layout.
   Fixed the same way that entry fixed it: apply orientation to every raw
   frame right after reading, and reorient the FFC field itself too (it's
   built by averaging raw-space frames, so it needs the same reorientation
   to line up with now-oriented frames -- transpose/flip is a fixed
   per-pixel relabelling, so it commutes with the per-pixel averaging,
   Gaussian smoothing, and normalization that build it, confirmed in that
   same prior entry as a valid equivalence).
2. **The first-pass FFC field (built from boundary FOVs) was visibly
   inhomogeneous** -- section 6's own histogram overlay already showed why:
   a large fraction of "boundary" (grid-exterior) FOVs are not actually
   tissue-free, so the field partly captured real anatomy instead of pure
   illumination/vignette. Fixed per the user's own suggestion: build the
   FFC field from **internal FOVs with near-100% real tissue coverage**
   instead. Iterated twice more on exactly how to pool interior FOVs
   (see `prompt_history/` for the full sequence: a 10-z average per FOV,
   then a single best-coverage frame per FOV, then every interior FOV's
   full-stack median) before a **dedicated comparison notebook**
   (`notebooks/tests/calculate_ffc/01_compare_ffc_methods.ipynb`) tested
   median/max/min projections against each other, smoothed and unsmoothed
   -- min projection won clearly (cleanest, most radially-symmetric field,
   no residual contamination bump, no smoothing needed), matching direct
   physical reasoning: a nucleus occupies only a handful of a FOV's 101
   z-planes at any given pixel, so the per-pixel **minimum** across the
   whole stack is overwhelmingly likely to be pure background. This
   notebook now uses that result: every interior FOV's own full-z-stack
   min projection.

## 1 -- Setup

In [ ]:
%matplotlib inline
# %matplotlib widget  # uncomment for interactive pan/zoom (ipympl)

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from skimage.measure import block_reduce
from PIL import Image, ImageDraw, ImageFont

# notebooks/tests/<subfolder>/ is three levels under the repo root (MERci/),
# same convention as notebooks/before_imaging/regular/ (3 levels).
MERCI_DIR = Path(os.getcwd()).parent.parent.parent
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config import ExperimentConfig
from MERci.common.metadata import ExperimentMetadata
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.common.io import read_image_frames
from MERci.acquisition.configs import get_fov_geometry, get_color_frame_indices
from MERci.acquisition.positions import find_exterior_fovs
from MERci.acquisition.configs import load_microscope_orientation, apply_microscope_orientation
from MERci.analysis.ffc import (
    select_ffc_exterior_fovs, compute_ffc_field_for_color, compute_mosaic_crop_px,
    save_ffc_field, load_ffc_field, apply_ffc,
)
from MERci.progress_display import ProgressReporter
from MERci.plots.experiment_plots import get_merci_figures_dir

NOTEBOOK_NAME = "elevation_heatmap"

# NOTEBOOK_GUIDELINES.md #5 -- explicit plot font sizes, reused by every
# plotting cell below instead of matplotlib's figsize-relative defaults.
PLOT_TITLE_FONTSIZE, PLOT_LABEL_FONTSIZE = 13, 11
PLOT_TICK_FONTSIZE, PLOT_LEGEND_FONTSIZE = 10, 10

## 2 -- Parameters

In [ ]:
# Same real dataset as notebooks/tests/decrease_fov_number/01_*/02_* -- see
# those notebooks' own Parameters cells for why SAMPLE_DIR is set explicitly
# (too large to copy locally) rather than auto-detected from this repo's
# own location.
SAMPLE_DIR = Path("/n/holylfs05/LABS/zhuang_lab/Lab/shared/projects/lineage_tracing/experiments/LT066_sample_01/merfish")

MICROSCOPE = "ST2"
OBJECTIVE  = "60X"
CHANNEL_NM = 405.0   # DAPI

DOWNSAMPLE_FACTOR = 8   # 2304 / 8 = 288 px

# THRESHOLD estimation (same convention as misc/measure_tissue_thickness_test.ipynb
# section 5 / analysis.fov.compute_tissue_fraction): the highest pixel value
# observed among the N_BACKGROUND_FRAMES lowest-mean boundary-FOV frames, in
# the SAME FFC-corrected + downsampled space the per-FOV elevation loop
# actually thresholds in.
N_BACKGROUND_FRAMES = 20

# FFC field v2 (every interior FOV's own full-z-stack min projection --
# see section 6's own markdown for why min, and why no coverage screening
# is needed).

# Representative block for the per-FOV elevation-matrix step (see the
# Scope note above) -- a compact grid_rows x grid_cols window of the real
# FOV grid, searched for one that both meets MIN_BLOCK_FOVS and contains a
# boundary-FOV count in EXT_COUNT_RANGE (i.e. straddles a real tissue edge
# rather than sitting purely in the interior or purely on the perimeter).
BLOCK_GRID_ROWS  = 5
BLOCK_GRID_COLS  = 5
MIN_BLOCK_FOVS   = 15
EXT_COUNT_RANGE  = (3, 10)

# z-sweep GIF (same idiom as misc/measure_tissue_thickness_test.ipynb section 24)
GIF_Z_STRIDE          = 5     # every Nth z-step
GIF_FRAME_DURATION_MS = 300
# No separate GIF display-width constant -- each frame is saved at the
# stitched canvas's own native resolution (set by DOWNSAMPLE_FACTOR/the
# block size above), not resized to an unrelated fixed width.

CACHE_DIR = SAMPLE_DIR / "analysis" / "cache" / NOTEBOOK_NAME
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR = get_merci_figures_dir(SAMPLE_DIR, "tests", NOTEBOOK_NAME, subfolder="tissue_thickness")

# The real experiment's own identity (via its deployed MERci/ clone under
# SAMPLE_DIR), NOT this repo's own location -- this notebook is not deployed
# inside SAMPLE_DIR (see the Scope note above), so MERCI_DIR resolves this
# repo's own identity instead, same distinction
# notebooks/tests/decrease_fov_number/02_*.ipynb already makes.
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(SAMPLE_DIR / "MERci")
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)

pixel_size_um, image_size_px = get_fov_geometry(MICROSCOPE, OBJECTIVE)

config = ExperimentConfig.from_sample_dir(
    SAMPLE_DIR,
    positions_txt=SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}.txt",
    image_suffix=".zarr", microscope=MICROSCOPE,
    pixel_size_um=pixel_size_um, image_size_px=image_size_px,
)
meta = ExperimentMetadata.load(config.round_info_csv, config.positions_txt, config.data_dir,
                                image_suffix=config.image_suffix)

# This repo's own bundled MERlin microscope-parameters JSONs (MERci/data/configs/
# merlin/microscope/) -- same MICROSCOPE_ORIENTATION_DIR convention as
# misc/measure_tissue_thickness_test.ipynb -- resolve once, apply to every raw
# frame read below (see the Review note above).
MICROSCOPE_ORIENTATION_DIR = MERCI_DIR / "data" / "configs" / "merlin" / "microscope"
MICROSCOPE_ORIENTATION = load_microscope_orientation(MICROSCOPE, MICROSCOPE_ORIENTATION_DIR)

cells_round_id = meta.round_for_imaging_type("cells")
round_info = meta.rounds[cells_round_id]

# {fov_id: (x, y)} -- scoped to THIS round's own real imaged FOVs (matches
# select_ffc_exterior_fovs's own internal restriction), not the raw
# experiment-wide positions.txt, so transit-only FOVs never enter the set.
positions = {fov_id: meta.fovs[fov_id].position
             for fov_id in round_info.fov_files if round_info.fov_files[fov_id]}

for s in meta.series_for_round(cells_round_id):
    if s.hal_config:
        from MERci.acquisition.configs import find_frame_table_for_hal_config
        ft_path = find_frame_table_for_hal_config(config.settings_dir / s.hal_config, config.metadata_dir)
        break
frame_table = pd.read_csv(ft_path, index_col=0)

channel_frames  = frame_table[frame_table["color"].round(0) == round(CHANNEL_NM)].sort_values("z")
z_frame_indices = channel_frames.index.tolist()
z_um_values     = channel_frames["z"].tolist()
mid_frame_idx   = get_color_frame_indices(frame_table)[CHANNEL_NM]

def z_frame_index_for(z_target, tol=1e-6):
    for idx, z in zip(z_frame_indices, z_um_values):
        if abs(z - z_target) < tol:
            return int(idx)
    raise ValueError(f"No exact frame at z={z_target} um in this round's {CHANNEL_NM} nm z-grid")

print(f"SAMPLE_DIR   : {SAMPLE_DIR}")
print(f"cells round  : {cells_round_id}  ({len(positions)} FOV(s) with real files)")
print(f"step_size_um : {config.step_size_um:.2f}   image_size_px: {config.image_size_px}")
print(f"{CHANNEL_NM} nm: {len(z_frame_indices)} z-plane(s), {z_um_values[0]:.1f}-{z_um_values[-1]:.1f} um, mid-z frame_idx={mid_frame_idx}")
print(f"Microscope orientation ({MICROSCOPE}): {MICROSCOPE_ORIENTATION}")
print(f"Cache  : {CACHE_DIR}")
print(f"Figures: {FIGURES_DIR}")

## 3 -- Identify boundary FOVs

"Boundary FOVs" = the *exterior* FOVs of the imaged grid (outer perimeter +
any hole edges) -- `find_exterior_fovs`, the same definition
`analysis/ffc.py`'s `"exterior_grid"` FFC-candidate strategy already uses.
Used below only as a bootstrap for a rough background threshold (section
5) and to pick the representative block (section 8) -- the real FFC field
(section 6) is built from *interior* FOVs instead (see the Review note).

In [ ]:
boundary_fov_ids  = find_exterior_fovs(
    positions, config.step_size_um,
    connectivity=config.ffc_connectivity, tolerance_fraction=config.ffc_neighbor_tolerance,
)
interior_fov_ids = sorted(set(positions) - boundary_fov_ids)
print(f"{len(boundary_fov_ids)} / {len(positions)} FOVs are boundary (exterior-grid) FOVs")
print(f"{len(interior_fov_ids)} / {len(positions)} FOVs are interior FOVs")

grid_indices = {}
xy = np.array([positions[i] for i in positions])
x0, y0 = xy[:, 0].min(), xy[:, 1].min()
for fov_id, (x, y) in positions.items():
    col = int(round((x - x0) / config.step_size_um))
    row = int(round((y - y0) / config.step_size_um))
    grid_indices[fov_id] = (row, col)

## 4 -- Plot: boundary vs. interior FOVs

Same plot idiom as `02_create_positions_from_boundaries.ipynb`'s "FOV
layout" plot (one rectangle per FOV footprint at its real stage
position).

In [ ]:
half = config.step_size_um / config.non_overlap_fraction / 2   # true FOV footprint half-width

def plot_fov_highlight(highlight_ids, highlight_label, highlight_color, title, figure_name):
    fig, ax = plt.subplots(figsize=(8, 7))
    for fov_id, (x, y) in positions.items():
        is_hl = fov_id in highlight_ids
        ax.add_patch(mpatches.Rectangle(
            (x - half, y - half), 2 * half, 2 * half,
            lw=0.3, edgecolor=highlight_color if is_hl else "0.6",
            facecolor=highlight_color if is_hl else "0.6",
            alpha=0.6 if is_hl else 0.15,
        ))
    ax.plot([], [], "s", color=highlight_color, alpha=0.6, ms=8, label=f"{highlight_label} ({len(highlight_ids)})")
    ax.plot([], [], "s", color="0.6", alpha=0.3, ms=8, label=f"other ({len(positions) - len(highlight_ids)})")
    ax.invert_yaxis(); ax.axis("equal")
    ax.set_title(title, fontsize=PLOT_TITLE_FONTSIZE)
    ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
    ax.legend(fontsize=PLOT_LEGEND_FONTSIZE, loc="best")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.{figure_name}.png", dpi=150)
    plt.show()

plot_fov_highlight(boundary_fov_ids, "boundary (exterior-grid)", "tab:red",
                    f"Boundary vs. interior FOVs -- {SAMPLE_NAME}", "boundary_fovs")

## 5 -- FFC field v1 (boundary FOVs) -- comparison reference only

One mid-z frame per boundary FOV (`select_ffc_exterior_fovs`) feeds a
first-pass FFC field (`compute_ffc_field_for_color`), reoriented right
after computing (see the Review note -- the field is built from raw-space
frames, so it needs the same `apply_microscope_orientation` transform the
frames it will divide get). Kept only as a point of comparison against the
real field built in section 6 below.

In [ ]:
def ffc_correct_and_downsample(raw_frame, ffc_field, factor):
    oriented = apply_microscope_orientation(raw_frame, **MICROSCOPE_ORIENTATION)
    corrected = apply_ffc(oriented, ffc_field)
    return block_reduce(corrected, (factor, factor), func=np.mean)

def estimate_threshold(ds_images, n_background_frames):
    '''Same convention as misc/measure_tissue_thickness_test.ipynb section 5 /
    analysis.fov.compute_tissue_fraction: the highest pixel value observed
    among the N lowest-mean frames -- the highest value that can plausibly
    occur as background noise, in whatever (FFC-corrected + downsampled)
    space *ds_images* is already in.'''
    means = {fov_id: float(ds.mean()) for fov_id, ds in ds_images.items()}
    lowest_mean_ids = sorted(means, key=means.get)[:n_background_frames]
    return max(float(ds_images[fov_id].max()) for fov_id in lowest_mean_ids)

ffc_v1_cache_path = CACHE_DIR / f"ffc_field_v1_boundary_{int(CHANNEL_NM)}nm.npz"
if ffc_v1_cache_path.exists():
    ffc_field_v1_boundary, _ = load_ffc_field(ffc_v1_cache_path)
    print(f"Loaded cached v1 (boundary) FFC field: {ffc_v1_cache_path}")
else:
    boundary_samples = select_ffc_exterior_fovs(cells_round_id, config, meta, mid_frame_idx)
    print(f"Computing v1 (comparison) FFC field from {len(boundary_samples)} boundary-FOV frame(s) ...")
    ffc_field_v1_boundary, meta_v1 = compute_ffc_field_for_color(
        boundary_samples, smooth_sigma_px=config.ffc_smooth_sigma_px,
        normalize_percentile=config.ffc_normalize_percentile, ffc_min_value=config.ffc_min_value,
    )
    ffc_field_v1_boundary = apply_microscope_orientation(ffc_field_v1_boundary, **MICROSCOPE_ORIENTATION)
    save_ffc_field(ffc_v1_cache_path, ffc_field_v1_boundary, meta_v1)
    print(f"Saved: {ffc_v1_cache_path}  ({meta_v1})")

## 6 -- FFC field v2: every interior FOV's own full-z-stack MIN projection

Per `notebooks/tests/calculate_ffc/01_compare_ffc_methods.ipynb`'s own
real comparison (median/max/min x smoothed/unsmoothed, same 882-interior-
FOV population): **min projection won clearly** -- the cleanest, most
radially-symmetric field, no residual contamination bump, no smoothing
needed to look clean. Matches direct physical reasoning: a nucleus only
occupies a handful of a FOV's 101 z-planes at any given pixel, so the
per-pixel **minimum** across the whole stack is overwhelmingly likely to
be a pure background reading -- more robust to real tissue signal than the
median this notebook originally used (which only needs a MINORITY of
z-planes to carry signal to resist contamination, but can still be pulled
up), and far more robust than a max (which is nearly guaranteed to catch
real signal wherever any nucleus passes through). No coverage screening --
every interior FOV (all `len(interior_fov_ids)` of them) contributes its
own min projection.

**Reading every interior FOV's full z-stack serially would take ~3 hours**
at the per-frame rate seen so far -- submitted instead as a SLURM array
job (`cli_compute_fov_projections.py` + `cluster_submit.build_fov_
projections_array_script`, the same array-job pattern `misc/measure_
tissue_thickness_test.ipynb` already uses for similarly heavy per-FOV
z-stack loops), one task per FOV, each reading only its own 101 frames.
Re-run this cell later (after the job finishes) to pick up newly-written
per-FOV min projections; the next cell loads them and builds the field
once every FOV's is ready.

In [ ]:
fov_min_dir = CACHE_DIR / "fov_min"
fov_min_dir.mkdir(parents=True, exist_ok=True)

def fov_min_path(fov_id):
    return fov_min_dir / f"fov{fov_id:04d}_min.npy"

to_compute_min = [f for f in interior_fov_ids if not fov_min_path(f).exists()]
print(f"{len(interior_fov_ids) - len(to_compute_min)} / {len(interior_fov_ids)} interior FOV "
      f"min projection(s) already cached; {len(to_compute_min)} more needed.")

USE_SLURM_ARRAY         = True   # set False to compute locally/serially instead (~3h for all interior FOVs)
SLURM_ARRAY_CONCURRENCY = 50
SLURM_MEM               = "8gb"   # 101 full-res frames in memory at once (~1 GB raw + working copies)
SLURM_TIME              = "00:15:00"

if to_compute_min and USE_SLURM_ARRAY:
    import csv
    import json
    from MERci.acquisition.cluster_submit import build_fov_projections_array_script, submit_sbatch, is_job_active

    min_job_sentinel = CACHE_DIR / "fov_min_job.json"
    cached_min_job = json.loads(min_job_sentinel.read_text()) if min_job_sentinel.exists() else None

    if (cached_min_job is not None and cached_min_job.get("n_pending") == len(to_compute_min)
            and is_job_active(cached_min_job["job_id"])):
        print(f"SLURM array job {cached_min_job['job_id']} is still active "
              f"({len(to_compute_min)} FOV(s) pending) -- re-run this cell later once it finishes.")
    else:
        manifest_path = CACHE_DIR / "fov_min_manifest.csv"
        with open(manifest_path, "w", newline="") as fh:
            writer = csv.writer(fh)
            writer.writerow(["fov_id", "image_path"])
            for fov_id in to_compute_min:
                writer.writerow([fov_id, round_info.fov_files[fov_id][0]])

        script_path = CACHE_DIR / "fov_min.sh"
        build_fov_projections_array_script(
            sample_dir=SAMPLE_DIR, manifest_path=manifest_path, output_dir=fov_min_dir,
            frame_indices=z_frame_indices, statistics=["min"], orientation=MICROSCOPE_ORIENTATION,
            n_pending=len(to_compute_min), output_path=script_path,
            array_concurrency=SLURM_ARRAY_CONCURRENCY, mem=SLURM_MEM, time=SLURM_TIME,
        )
        job_id = submit_sbatch(script_path)
        if job_id is not None:
            min_job_sentinel.write_text(json.dumps({"job_id": job_id, "n_pending": len(to_compute_min)}))
            print(f"Submitted SLURM array job {job_id} for {len(to_compute_min)} FOV(s) -- "
                  f"re-run this cell later once it finishes to load the results.")
        else:
            print("sbatch submission failed (see the logged error above) -- fix the issue and re-run this cell.")
elif to_compute_min:
    reporter = ProgressReporter(total=len(to_compute_min), label="Computing per-FOV min projections (local)")
    for fov_id in reporter.wrap(to_compute_min):
        fpath = round_info.fov_files[fov_id][0]
        stack = read_image_frames(fpath, z_frame_indices).astype(np.float32)
        min_img = np.min(stack, axis=0)
        min_img = apply_microscope_orientation(min_img, **MICROSCOPE_ORIENTATION)
        np.save(fov_min_path(fov_id), min_img.astype(np.float32))

In [ ]:
from scipy.ndimage import gaussian_filter

fov_min_ready = [f for f in interior_fov_ids if fov_min_path(f).exists()]
print(f"{len(fov_min_ready)} / {len(interior_fov_ids)} interior FOV min projection(s) available.")

if len(fov_min_ready) < len(interior_fov_ids):
    print("Still waiting on the SLURM array job (or USE_SLURM_ARRAY=False loop) above -- "
          "re-run both this cell and the previous one once every FOV's min projection is ready.")
else:
    ffc_v2_cache_path = CACHE_DIR / f"ffc_field_v2_interior_min_{int(CHANNEL_NM)}nm.npz"
    if ffc_v2_cache_path.exists():
        ffc_field, ffc_meta = load_ffc_field(ffc_v2_cache_path)
        print(f"Loaded cached v2 (interior, min) FFC field: {ffc_v2_cache_path}  ({ffc_meta})")
    else:
        def build_ffc_field_from_image_paths(paths, smooth_sigma_px, normalize_percentile, ffc_min_value):
            '''Same recipe as analysis.ffc.compute_ffc_field_for_color's own tail (mean ->
            Gaussian smooth -> percentile-normalize -> floor-clip), applied to already-computed
            per-FOV images (already oriented) instead of raw disk samples it reads itself --
            streamed one at a time (like that function's own running-total loop) rather than
            loading every one of the (here, hundreds of) images into memory at once.'''
            total = None
            for p in paths:
                img = np.load(p).astype(np.float64)
                total = img if total is None else total + img
            field = (total / len(paths)).astype(np.float32)
            field = gaussian_filter(field, sigma=smooth_sigma_px)
            norm_value = np.percentile(field, normalize_percentile)
            if norm_value > 0:
                field = field / norm_value
            return np.clip(field, ffc_min_value, None).astype(np.float32)

        min_paths = [fov_min_path(f) for f in interior_fov_ids]
        ffc_field = build_ffc_field_from_image_paths(
            min_paths, config.ffc_smooth_sigma_px, config.ffc_normalize_percentile, config.ffc_min_value,
        )
        ffc_meta = {"n_samples": len(min_paths), "statistic": "min", "smooth_sigma_px": config.ffc_smooth_sigma_px,
                    "normalize_percentile": config.ffc_normalize_percentile, "ffc_min_value": config.ffc_min_value}
        save_ffc_field(ffc_v2_cache_path, ffc_field, ffc_meta)
        print(f"Saved: {ffc_v2_cache_path}  ({ffc_meta})")

In [ ]:
if "ffc_field" not in dir():
    print("FFC field v2 not built yet (still waiting on the SLURM array job above) -- skipping.")
else:
    example_ids = interior_fov_ids[:6]
    example_images = {f: np.load(fov_min_path(f)) for f in example_ids}
    fig, axes = plt.subplots(1, len(example_ids), figsize=(3 * len(example_ids), 3.2))
    vmax = np.percentile(np.concatenate([img.ravel() for img in example_images.values()]), 99.5)
    for ax, fov_id in zip(np.atleast_1d(axes), example_ids):
        ax.imshow(example_images[fov_id], cmap="gray", vmin=0, vmax=vmax)
        ax.set_title(f"FOV {fov_id}", fontsize=PLOT_TITLE_FONTSIZE)
        ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
    fig.suptitle("Example interior-FOV full-stack min-projection images", fontsize=PLOT_TITLE_FONTSIZE)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.ffc_v2_candidate_averages.png", dpi=150)
    plt.show()

    cv_v1 = float(ffc_field_v1_boundary.std() / ffc_field_v1_boundary.mean())
    cv_v2 = float(ffc_field.std() / ffc_field.mean())

    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    for ax, field, label, cv in zip(axes, [ffc_field_v1_boundary, ffc_field], ["v1 (boundary)", "v2 (interior, fixed)"], [cv_v1, cv_v2]):
        im = ax.imshow(field, cmap="viridis", vmin=0, vmax=1.05)
        ax.set_title(f"{label}\ncoeff. of variation = {cv:.3f}", fontsize=PLOT_TITLE_FONTSIZE)
        ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
    fig.colorbar(im, ax=axes, shrink=0.8)
    fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.ffc_field_v1_vs_v2.png", dpi=150)
    plt.show()
    print(f"Field homogeneity (lower = flatter): v1 CV={cv_v1:.3f}   v2 CV={cv_v2:.3f}")

In [ ]:
def field_profiles(field):
    h, w = field.shape
    return field[h // 2, :], field[:, w // 2], np.diagonal(field)

if "ffc_field" not in dir():
    print("FFC field v2 not built yet (still waiting on the SLURM array job above) -- skipping.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
    for ax, field, label in zip(axes, [ffc_field_v1_boundary, ffc_field], ["v1 (boundary)", "v2 (interior, min)"]):
        row_profile, col_profile, diag_profile = field_profiles(field)
        ax.plot(row_profile, label="horizontal (row = H/2)")
        ax.plot(col_profile, label="vertical (col = W/2)")
        ax.plot(diag_profile, label="diagonal")
        ax.set_title(label, fontsize=PLOT_TITLE_FONTSIZE)
        ax.set_xlabel("pixel index along profile", fontsize=PLOT_LABEL_FONTSIZE)
        ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
        ax.legend(fontsize=PLOT_LEGEND_FONTSIZE)
    axes[0].set_ylabel("FFC field value", fontsize=PLOT_LABEL_FONTSIZE)
    fig.suptitle("FFC field 1D profiles (x/y/diagonal cross-sections)", fontsize=PLOT_TITLE_FONTSIZE)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.ffc_field_profiles.png", dpi=150)
    plt.show()

## 7 -- Final background/foreground threshold (v2 space)

Same estimator as section 5, recomputed against boundary-FOV frames
corrected with the real (v2) FFC field, plus the log-intensity overlay
(same idiom as `acquisition.mosaic.plot_tile_intensity_histograms`, reused
by `misc/measure_tissue_thickness_test.ipynb` section 5): every boundary
FOV's own histogram as a thin line, plus a bold pooled histogram.

In [ ]:
boundary_ds_path = CACHE_DIR / f"boundary_fov_downsampled_v2_{int(CHANNEL_NM)}nm.npz"
if boundary_ds_path.exists():
    _npz = np.load(boundary_ds_path)
    boundary_ds = {int(k.split("_")[1]): _npz[k] for k in _npz.files}
    print(f"Loaded {len(boundary_ds)} cached boundary-FOV downsampled frame(s): {boundary_ds_path}")
else:
    boundary_ds = {}
    reporter = ProgressReporter(total=len(boundary_fov_ids), label="[v2] Reading+correcting boundary FOVs")
    for fov_id in reporter.wrap(sorted(boundary_fov_ids)):
        fpath = round_info.fov_files[fov_id][0]
        raw = read_image_frames(fpath, [mid_frame_idx])[0]
        boundary_ds[fov_id] = ffc_correct_and_downsample(raw, ffc_field, DOWNSAMPLE_FACTOR).astype(np.float32)
    np.savez_compressed(boundary_ds_path, **{f"fov_{k}": v for k, v in boundary_ds.items()})
    print(f"Saved: {boundary_ds_path}")

THRESHOLD = estimate_threshold(boundary_ds, N_BACKGROUND_FRAMES)
print(f"THRESHOLD = {THRESHOLD:.1f} (max pixel value among the "
      f"{N_BACKGROUND_FRAMES} lowest-mean boundary FOVs, v2 FFC-corrected + downsampled space)")

In [ ]:
LOG_BINS = 200
all_vals = np.concatenate([np.clip(ds, 1, None).ravel() for ds in boundary_ds.values()])
bin_edges = np.linspace(np.log10(all_vals.min()), np.log10(all_vals.max()), LOG_BINS + 1)

fig, ax = plt.subplots(figsize=(8, 5))
pooled_hist = np.zeros(LOG_BINS)
for fov_id, ds in boundary_ds.items():
    hist, _ = np.histogram(np.log10(np.clip(ds, 1, None)), bins=bin_edges, density=True)
    ax.plot(bin_edges[:-1], hist, "-", lw=0.6, color="0.6", alpha=0.5)
    pooled_hist += hist
pooled_hist /= len(boundary_ds)
ax.plot(bin_edges[:-1], pooled_hist, "-", lw=2, color="k", label="pooled (mean over boundary FOVs)")
ax.axvline(np.log10(max(THRESHOLD, 1)), color="tab:red", ls="--",
           label=f"THRESHOLD = {THRESHOLD:.0f}")
ax.set_xlabel("log10(intensity)  [v2 FFC-corrected, downsampled]", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_ylabel("density", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_title(f"Boundary-FOV DAPI intensity overlay ({len(boundary_ds)} FOVs)", fontsize=PLOT_TITLE_FONTSIZE)
ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
ax.legend(fontsize=PLOT_LEGEND_FONTSIZE)
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.boundary_fov_histograms.png", dpi=150)
plt.show()

## 8 -- Pick a representative FOV block

Per this notebook's scope note (section intro), the per-z elevation loop
(section 9) runs on one compact block of the real grid rather than the
full 1138-FOV grid. The block is searched for automatically: the smallest
`BLOCK_GRID_ROWS x BLOCK_GRID_COLS` window (row-major scan) with at least
`MIN_BLOCK_FOVS` real FOVs and a boundary-FOV count inside
`EXT_COUNT_RANGE` -- i.e. one that straddles a real tissue edge (mix of
boundary + interior FOVs), rather than sitting purely in the interior or
purely on the perimeter, so it exercises both the FFC boundary-FOV logic
and the crop/stitch-across-a-real-edge logic.

In [ ]:
def pick_compact_block(grid_indices, boundary_ids, n_rows, n_cols, min_fovs, ext_range):
    rows = [r for r, c in grid_indices.values()]
    cols = [c for r, c in grid_indices.values()]
    for r0 in range(min(rows), max(rows) - n_rows + 2):
        for c0 in range(min(cols), max(cols) - n_cols + 2):
            block = [fid for fid, (r, c) in grid_indices.items()
                     if r0 <= r < r0 + n_rows and c0 <= c < c0 + n_cols]
            if len(block) < min_fovs:
                continue
            n_ext = sum(1 for f in block if f in boundary_ids)
            if ext_range[0] <= n_ext <= ext_range[1]:
                return r0, c0, sorted(block)
    raise RuntimeError("No block satisfies the given constraints -- widen BLOCK_GRID_ROWS/COLS, "
                        "MIN_BLOCK_FOVS, or EXT_COUNT_RANGE.")

block_r0, block_c0, block_fov_ids = pick_compact_block(
    grid_indices, boundary_fov_ids, BLOCK_GRID_ROWS, BLOCK_GRID_COLS, MIN_BLOCK_FOVS, EXT_COUNT_RANGE,
)
block_n_ext = sum(1 for f in block_fov_ids if f in boundary_fov_ids)
print(f"Block: rows [{block_r0}, {block_r0 + BLOCK_GRID_ROWS}), cols [{block_c0}, {block_c0 + BLOCK_GRID_COLS})")
print(f"{len(block_fov_ids)} real FOV(s), {block_n_ext} boundary FOV(s): {block_fov_ids}")

plot_fov_highlight(set(block_fov_ids), "section 8 block", "tab:blue",
                    f"Representative block location -- {SAMPLE_NAME}", "block_location")

## 9 -- Per-FOV elevation matrix `M` + downsampled DAPI z-stack

For every FOV in the block, and every z-plane: FFC-correct (v2 field),
downsample, threshold (`>= THRESHOLD`), and overwrite `M[i, j] = z_um`
wherever the downsampled pixel is foreground -- iterating z ascending, `M`
ends up holding each pixel's *topmost* foreground z (in µm). `M` starts at
0 (no sentinel needed: this round's z values start at 0.5 µm, never 0, so
0 unambiguously means "never foreground at any z"). Both `M` and the
downsampled corrected z-stack (reused by the GIF in section 11) are cached
per FOV -- NOTEBOOK_GUIDELINES.md #2/#3, skip-if-cached checked
individually per FOV.

In [ ]:
elevation_dir  = CACHE_DIR / "elevation"
downsampled_dir = CACHE_DIR / "downsampled_stack"
elevation_dir.mkdir(parents=True, exist_ok=True)
downsampled_dir.mkdir(parents=True, exist_ok=True)

def elevation_path(fov_id):
    return elevation_dir / f"fov_{fov_id:04d}.npy"

def downsampled_path(fov_id):
    return downsampled_dir / f"fov_{fov_id:04d}.npz"

def compute_fov_elevation(fpath, z_frame_indices, z_um_values, ffc_field, threshold, factor):
    M = None
    ds_stack = []
    for idx, z_um in zip(z_frame_indices, z_um_values):
        raw = read_image_frames(fpath, [int(idx)])[0]
        ds = ffc_correct_and_downsample(raw, ffc_field, factor).astype(np.float32)
        if M is None:
            M = np.zeros(ds.shape, dtype=np.float32)
        M[ds >= threshold] = z_um   # ascending z -> ends as the topmost foreground z
        ds_stack.append(ds)
    return M, np.stack(ds_stack, axis=0)

to_compute = [f for f in block_fov_ids if not (elevation_path(f).exists() and downsampled_path(f).exists())]
print(f"{len(block_fov_ids) - len(to_compute)} / {len(block_fov_ids)} FOV(s) already cached; "
      f"computing {len(to_compute)} more ({len(z_frame_indices)} z-plane(s) each).")

if to_compute:
    reporter = ProgressReporter(total=len(to_compute), label="Computing per-FOV elevation matrices")
    for fov_id in reporter.wrap(to_compute):
        fpath = round_info.fov_files[fov_id][0]
        M, ds_stack = compute_fov_elevation(fpath, z_frame_indices, z_um_values, ffc_field, THRESHOLD, DOWNSAMPLE_FACTOR)
        np.save(elevation_path(fov_id), M)
        np.savez_compressed(downsampled_path(fov_id), stack=ds_stack, z_um=np.asarray(z_um_values, dtype=np.float32))

elevation_matrices = {f: np.load(elevation_path(f)) for f in block_fov_ids}
print(f"Elevation matrix shape (per FOV): {next(iter(elevation_matrices.values())).shape}")

In [ ]:
example_ids = block_fov_ids[:6]
fig, axes = plt.subplots(1, len(example_ids), figsize=(3 * len(example_ids), 3.2))
vmax = max(float(elevation_matrices[f].max()) for f in example_ids)
for ax, fov_id in zip(np.atleast_1d(axes), example_ids):
    im = ax.imshow(elevation_matrices[fov_id], cmap="viridis", vmin=0, vmax=vmax)
    ax.set_title(f"FOV {fov_id}", fontsize=PLOT_TITLE_FONTSIZE)
    ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
cbar = fig.colorbar(im, ax=np.atleast_1d(axes).tolist(), shrink=0.8, label="elevation (um)")
cbar.ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
fig.suptitle("Example per-FOV elevation matrices (pre-crop)", fontsize=PLOT_TITLE_FONTSIZE)
fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.example_fov_elevation.png", dpi=150)
plt.show()

## 10 -- Stitch: crop overlap + assemble the block's elevation heatmap

Each FOV's downsampled `M` is centrally cropped to its non-overlap
footprint (`analysis.ffc.compute_mosaic_crop_px`, the same overlap-crop
convention `create_mosaic_ffc` already uses for round mosaics, converted
from raw to downsampled pixels), then placed into a grid-indexed canvas
sized `(block_rows * crop, block_cols * crop)` -- missing grid cells (no
real FOV, e.g. a hole) are left as `NaN`.

In [ ]:
crop_px_raw  = compute_mosaic_crop_px(config)
crop_px      = crop_px_raw // DOWNSAMPLE_FACTOR
ds_size      = image_size_px // DOWNSAMPLE_FACTOR
tile_size    = ds_size - 2 * crop_px
print(f"Raw overlap crop: {crop_px_raw} px/side -> downsampled: {crop_px} px/side -> tile size {tile_size}x{tile_size}")

def center_crop(arr, crop_px):
    if crop_px == 0:
        return arr
    return arr[crop_px:-crop_px, crop_px:-crop_px]

def stitch_by_grid(tiles, grid_indices, r0, c0, n_rows, n_cols, crop_px, fill=np.nan):
    cropped = {f: center_crop(t, crop_px) for f, t in tiles.items()}
    tile_h, tile_w = next(iter(cropped.values())).shape
    canvas = np.full((n_rows * tile_h, n_cols * tile_w), fill, dtype=np.float32)
    for fov_id, tile in cropped.items():
        r, c = grid_indices[fov_id]
        rr, cc = r - r0, c - c0
        canvas[rr * tile_h:(rr + 1) * tile_h, cc * tile_w:(cc + 1) * tile_w] = tile
    return canvas

elevation_heatmap = stitch_by_grid(
    elevation_matrices, grid_indices, block_r0, block_c0, BLOCK_GRID_ROWS, BLOCK_GRID_COLS, crop_px,
)
np.save(CACHE_DIR / "elevation_heatmap_block.npy", elevation_heatmap)
print(f"Stitched elevation heatmap shape: {elevation_heatmap.shape}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
cmap = plt.cm.viridis.copy(); cmap.set_bad("0.85")
im = ax.imshow(np.ma.masked_invalid(elevation_heatmap), cmap=cmap)
ax.set_title(f"Tissue elevation heatmap -- {len(block_fov_ids)}-FOV block", fontsize=PLOT_TITLE_FONTSIZE)
ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
cbar = fig.colorbar(im, ax=ax, label="elevation (um)")
cbar.ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.elevation_heatmap.png", dpi=150)
plt.show()

## 11 -- Z-sweep GIF (downsampled, FFC-corrected DAPI mosaic)

Same block, same crop/stitch function as section 10, applied per z-step to
the raw (non-elevation) downsampled intensity stacks cached in section 9 --
one PNG per z-step, assembled into a GIF (same idiom as
`misc/measure_tissue_thickness_test.ipynb` section 24: shared intensity
scale across every frame, so brightness changes reflect real signal
fading, not per-frame auto-contrast). Each frame is saved at the stitched
canvas's own native resolution -- no separate resize step -- so
`DOWNSAMPLE_FACTOR` is the one thing controlling both compute cost and the
GIF's real size/sharpness, instead of a resize target disconnected from
it (an earlier version resized to a fixed width regardless of the
canvas's actual size, which could even upsample a small canvas at a high
downsample factor).

In [ ]:
downsampled_stacks = {f: np.load(downsampled_path(f))["stack"] for f in block_fov_ids}
n_z = next(iter(downsampled_stacks.values())).shape[0]
gif_z_positions = list(range(0, n_z, GIF_Z_STRIDE))

pooled_pixels = np.concatenate([stack[z].ravel() for stack in downsampled_stacks.values() for z in gif_z_positions])
vmin_g, vmax_g = np.percentile(pooled_pixels, [1.0, 99.0])
print(f"Shared GIF display scale (p1-p99): [{vmin_g:.0f}, {vmax_g:.0f}]")

def to_uint8(arr, vmin, vmax):
    scaled = (arr.astype(np.float64) - vmin) / max(vmax - vmin, 1e-9) * 255
    return np.clip(scaled, 0, 255).astype(np.uint8)

pil_frames = []
reporter = ProgressReporter(total=len(gif_z_positions), label="Assembling GIF frames")
for z_pos in reporter.wrap(gif_z_positions):
    tiles = {f: to_uint8(stack[z_pos], vmin_g, vmax_g) for f, stack in downsampled_stacks.items()}
    canvas = stitch_by_grid(tiles, grid_indices, block_r0, block_c0, BLOCK_GRID_ROWS, BLOCK_GRID_COLS, crop_px, fill=0)
    img = Image.fromarray(canvas.astype(np.uint8), mode="L")   # native canvas resolution -- no resize
    draw = ImageDraw.Draw(img)
    font = ImageFont.load_default(size=max(14, img.width // 40))
    draw.text((8, 8), f"z = {z_um_values[z_pos]:.1f} um", fill=255, font=font)
    pil_frames.append(img)

gif_path = FIGURES_DIR / f"{NOTEBOOK_NAME}_z_sweep_block.gif"
pil_frames[0].save(gif_path, save_all=True, append_images=pil_frames[1:],
                    duration=GIF_FRAME_DURATION_MS, loop=0)
print(f"Saved: {gif_path}  ({len(pil_frames)} frame(s), {pil_frames[0].size[0]}x{pil_frames[0].size[1]} px)")

## Summary

- Boundary (exterior-grid) FOVs identified over the real, full 1138-FOV
  grid: sections 3-4.
- FFC field built from every interior FOV's own full-z-stack min
  projection (per `notebooks/tests/calculate_ffc/01_compare_ffc_
  methods.ipynb`'s own real comparison against median/max, smoothed/
  unsmoothed), compared directly against the original boundary-FOV field:
  sections 5-6. Final threshold estimated in that field's own corrected
  space: section 7.
- Every raw frame (FFC-field samples, boundary-FOV samples, per-FOV
  elevation z-stacks) is reoriented (`apply_microscope_orientation`)
  before any further processing -- see the Review note at the top.
- Per-FOV elevation matrices, crop+stitch, and the z-sweep GIF validated on
  one real 5x5 representative block (straddling a real tissue edge):
  sections 8-11.
- Figures: `{FIGURES_DIR}`. Cache (elevation matrices, downsampled stacks,
  FFC fields, stitched heatmap): `{CACHE_DIR}`.

**Follow-up for another session** (per this investigation's own scope):
promote `compute_fov_elevation`/`center_crop`/`stitch_by_grid`/the
internal-FOV FFC-selection logic into `MERci` proper (`analysis/` --
alongside `ffc.py`/`round.py`), and run section 9 over the real full
1138-FOV grid via a SLURM array job (same pattern as `misc/
measure_tissue_thickness_test.ipynb` sections 14/23/24), then wire the
result into `after_imaging/08_measure_tissue_thickness.ipynb`.